In [1]:
import sys, subprocess, pkgutil, os

def ensure_pyg():
    import torch
    if pkgutil.find_loader("torch_geometric") is not None:
        print("torch_geometric already available ✅")
        return
    ver = torch.__version__.split('+')[0]   # e.g. '2.9.0'
    cu  = torch.version.cuda               # e.g. '12.8'
    if cu is None:
        wheel_idx = f"torch-{ver}+cpu.html"
    else:
        cu_tag = "cu" + cu.replace('.', '')   # 'cu128'
        wheel_idx = f"torch-{ver}+{cu_tag}.html"
    idx_url = f"https://data.pyg.org/whl/{wheel_idx}"
    print("[INFO] Installing PyG wheels for", ver, cu, "from", idx_url)
    cmds = [
        [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
        [sys.executable, "-m", "pip", "install", "pyg_lib", "torch_scatter", "torch_sparse", "torch_cluster", "torch_spline_conv", "-f", idx_url],
        [sys.executable, "-m", "pip", "install", "torch_geometric"],
    ]
    for c in cmds:
        print("->", " ".join(c))
        subprocess.check_call(c)

try:
    ensure_pyg()
    import torch_geometric
    print("✅ torch_geometric version:", torch_geometric.__version__)
except Exception as e:
    print("❌ Failed to prepare torch_geometric:", e)
    print("  - カーネルが別環境の可能性があります。上のセル出力のURL/コマンドでターミナル側に先に入れてから、")
    print("    Notebookのカーネルをそのvenvに切り替えてください（Kernel → Change Kernel）。")
    raise



torch_geometric already available ✅


/home/yudai/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/requests/__init__.py:87: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (4.0.0) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


✅ torch_geometric version: 2.7.0


In [2]:
from pathlib import Path

in_dir = Path("/home/yudai/Project/research/Vul_Detection/data/input")
files = sorted(in_dir.glob("*_cpg_input.pkl"))
print("Found:", len(files))
for i, f in enumerate(files, 1):
    print(f"{i}. {f.name}")
assert files, "data/input/*_cpg_input.pkl が見つかりません。"


Found: 2
1. 0_cpg_input.pkl
2. 1_cpg_input.pkl


In [3]:
# --- shim: map numpy._core.* -> numpy.core.* for unpickling ---
import sys, types, importlib
import numpy as _np

_aliases = {
    "numpy._core": "numpy.core",
    "numpy._core.numeric": "numpy.core.numeric",
    "numpy._core.fromnumeric": "numpy.core.fromnumeric",
    "numpy._core.multiarray": "numpy.core.multiarray",
    "numpy._core._multiarray_umath": "numpy.core._multiarray_umath",
    "numpy._core.records": "numpy.core.records",
}

for new, old in _aliases.items():
    try:
        mod = importlib.import_module(old)
    except Exception:
        # 一部は存在しない場合があるが、numeric と fromnumeric があれば十分なことが多い
        continue
    parent = new.rsplit(".", 1)[0]
    if parent not in sys.modules:
        sys.modules[parent] = types.ModuleType(parent)
    sys.modules[new] = mod

print("✅ NumPy shim active. Now you can pd.read_pickle(...) safely.")


✅ NumPy shim active. Now you can pd.read_pickle(...) safely.


In [20]:
import pandas as pd
from torch_geometric.data import Data

pkl0 = files[0]
pkl1 = files[1]
df0 = pd.read_pickle(pkl0)
df1 = pd.read_pickle(pkl1)
print("Columns:", list(df0.columns), "shape:", df0.shape)
print("Columns:", list(df1.columns), "shape:", df1.shape)
display(df0.head(1))  # 先頭行プレビュー
display(df1.head(1))

assert {"input","target","func"}.issubset(df0.columns), "必要列（input/target/func）がありません。"
assert isinstance(df0.iloc[0]["input"], Data), "input が torch_geometric.data.Data ではありません。"

assert {"input","target","func"}.issubset(df1.columns), "必要列（input/target/func）がありません。"
assert isinstance(df1.iloc[0]["input"], Data), "input が torch_geometric.data.Data ではありません。"


Columns: ['input', 'target', 'func'] shape: (965, 3)
Columns: ['input', 'target', 'func'] shape: (990, 3)


,input,target,func
0,"[(x, [tensor([ 0.0000e+00, -1.4213e-01, 3.666...",1,"glue(cirrus_bitblt_rop_fwd_, ROP_NAME)(CirrusV..."


,input,target,func
0,"[(x, [tensor([ 0.0000e+00, -1.4198e-01, 3.666...",0,static uint64_t virtio_net_guest_offloads_by_f...
